#Projeto de Engenharia de dados - Pipeline  Medallion (OLX)
##Este projeto demonstra a construção  de um pipeline de engenharia de dados utilizando a arquitetura medallion(BRONZE, SILVER, GOLD).Os dados foram  extraídos  de anuncios de imóveis da OLX  e processamento  em databricks utilizando Pyspark.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, coalesce, lit, round, current_timestamp

#Extração dos dados  da Olx

##O dataset em formaato CSV é carregado para on databricks.O arquivo contém  anúncios  de imóveis  da OLX com informações  como titulo, preço, localização, área , qtd de quartos e demais caracteristicas.

In [0]:
df =  (

spark.read
.option("header", "true")
.option("inferSchema", "true")
.csv("/Volumes/workspace/default/dados/dataset_olx_raw.csv")
)



##BRONZE

#A camada bronze representa a ingestao dos dados brutos.
#Nesta etapa:
# Os Dados sao diretamente do arquivo csv
#Nenhuma transformação de negócio é aplicada 
#Os dados sao no formato delta lake.
# O objetivo é preservar a informação original para auditoria e processamento.

In [0]:
df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.default.bronze_olx")


#silver

#A camada Silver é responsável pelo tratamento e padronização dos dados.

#Transformações realizadas:

#Remoção de espaços em branco.

#Padronização de colunas textuais.

#Conversão dos tipos de dados.

#Tratamento de valores nulos.

#Remoção de registros sem informações essenciais.

#Eliminação de duplicidades.

#Preparação dos dados para análise.

In [0]:


#leitura_da_bronze
df_bronze = spark.table("workspace.default.bronze_olx")

#tratamento_camada_bronze

df_silver =(

    df_bronze

    #remover espaços extras
    .withColumn("titulo", F.trim(F.col("titulo")))
     .withColumn("url", F.trim(F.col("url")))
     .withColumn("tipo", F.trim(F.col("tipo")))
    .withColumn("bairro", F.trim(F.col("bairro")))
    .withColumn("cidade", F.trim(F.col("cidade")))
    .withColumn("estado", F.trim(F.col("estado")))
    .withColumn("cep", F.trim(F.col("cep")))
    .withColumn("descricao", F.trim(F.col("descricao")))

    #TRANSFORMAR TEXTOS VAZIOS EM NULL
    .replace("", None)


    #converter as colunas numericas
    .withColumn("preco", F.col("preco").cast("double"))
    .withColumn("quartos", F.col("quartos").cast("double"))
    .withColumn("banheiros", F.col("banheiros").cast("double"))
    .withColumn("garagens", F.col("garagens").cast("double"))
    .withColumn("area_m2", F.col("area_m2").cast("double"))

    #remover somente registros sem informação
    .na.drop(subset = ["titulo", "url"])

    #remove somente registros sem informação
    .dropDuplicates(["url"])

     )
                                

(

   df_silver.write\
    .mode("overwrite")\
    .saveAsTable("workspace.default.silver_real_estate_listings")
   
)


#Gold

#A camada Gold disponibiliza dados prontos para consumo analítico.

#Nesta etapa são criados indicadores de negócio como:

#Preço por metro quadrado.

#Classificação do imóvel por faixa de preço.

#Quantidade total de cômodos.

#Classificação da área do imóvel.

#Data de processamento

In [0]:


df_silver = spark.table("workspace.default.silver_real_estate_listings")
df_gold = (df_silver
.withColumn("preco_m2",
            when((col("area_m2").isNotNull()) & (col("area_m2") > 0),
                 round(col("preco") / col("area_m2"), 2))
)
.withColumn("categoria_preco",
            when(col("preco") < 300000, "baixo")
            .when(col("preco") < 700000, "medio")
            .otherwise("alto")
)
.withColumn("total_comodos",
            coalesce(col("quartos"), lit(0))
            + coalesce(col("banheiros"), lit(0))
)
.withColumn("Possui_garagem",
            when(col("garagens") > 0, "sim").otherwise("nao")
)
.withColumn("dt_carga",
            current_timestamp()
)
)

(

df_gold.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.default.gold_real_estate_metrics")
    )